In [ ]:
!pip install -q pydantic httpx tenacity openai python-dotenv
!mkdir -p copywriter data notebooks

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded.")
except Exception:
    print("No key set — that's fine, mock mode works without one.")

API key loaded.


In [ ]:
%%writefile copywriter/__init__.py
"""
copywriter — Automated Copywriting & Tone Transformer
"""
__version__ = "0.1.0"

Overwriting copywriter/__init__.py


In [ ]:
%%writefile copywriter/schemas.py
"""
Pydantic schemas for validating AI-generated marketing copy.
"""
from typing import List
from pydantic import BaseModel, field_validator

PLATFORM_RULES = {
    "LinkedIn": {"max_hashtags": 3, "max_chars": None},
    "Instagram": {"min_hashtags": 5, "max_hashtags": 10, "max_chars": None},
    "Email": {"max_hashtags": 0, "max_chars": None, "max_headline_chars": 50},
    "Twitter/X": {"max_hashtags": 2, "max_chars": 280},
}


class CopyOutput(BaseModel):
    """The required shape of every generated piece of copy.

    NOTE: field order matters. In Pydantic v2, a field validator's
    `info.data` only contains fields validated earlier (declared
    earlier). `platform` and `tone_used` go first so later
    validators can read them.
    """

    platform: str
    tone_used: str
    headline: str
    body: str
    hashtags: List[str] = []
    cta: str
    character_count: int

    @field_validator("character_count")
    @classmethod
    def check_character_limit(cls, v: int, info):
        platform = info.data.get("platform", "")
        max_chars = PLATFORM_RULES.get(platform, {}).get("max_chars")
        if max_chars and v > max_chars:
            raise ValueError(
                f"{platform} copy is {v} characters, which exceeds the "
                f"{max_chars}-character limit."
            )
        return v

    @field_validator("hashtags")
    @classmethod
    def check_hashtag_count(cls, v: List[str], info):
        platform = info.data.get("platform", "")
        rules = PLATFORM_RULES.get(platform, {})
        max_h, min_h = rules.get("max_hashtags"), rules.get("min_hashtags")

        if max_h is not None and len(v) > max_h:
            raise ValueError(
                f"{platform} copy has {len(v)} hashtags, max allowed is {max_h}."
            )
        if min_h is not None and len(v) < min_h:
            raise ValueError(
                f"{platform} copy has {len(v)} hashtags, minimum required is {min_h}."
            )
        return v

    @field_validator("headline")
    @classmethod
    def check_headline_length(cls, v: str, info):
        platform = info.data.get("platform", "")
        max_len = PLATFORM_RULES.get(platform, {}).get("max_headline_chars")
        if max_len and len(v) > max_len:
            raise ValueError(
                f"{platform} headline is {len(v)} characters, max allowed "
                f"is {max_len}."
            )
        return v

Overwriting copywriter/schemas.py


In [ ]:
%%writefile copywriter/templates.py
"""
The Master Instruction Template — the "gatekeeper" of the pipeline.
"""

VALID_PLATFORMS = ["LinkedIn", "Instagram", "Email", "Twitter/X"]
VALID_TONES = ["Professional", "Witty", "Formal", "Casual", "Urgent"]

BRAND_RULES = """\
BRAND VOICE RULES (never break these):
- Never use the words "revolutionary", "game-changer", "disruptive", or "synergy"
- Write in second person ("you", "your"), not first person
- Do not make unverifiable superlative claims (e.g. "the best in the world")
- Always end with exactly one clear call to action (CTA)\
"""

PLATFORM_INSTRUCTIONS = {
    "LinkedIn": (
        "FORMAT: Paragraph form (no bullet lists). Maximum 300 words. "
        "Include at most 3 relevant hashtags. The tone should feel like "
        "thought leadership."
    ),
    "Instagram": (
        "FORMAT: The first line must hook the reader before the 'more' "
        "fold. Use short sentences and line breaks. Maximum 150 words "
        "before hashtags. Include between 5 and 10 relevant hashtags."
    ),
    "Email": (
        "FORMAT: Provide a subject line (max 50 characters) as the "
        "headline, a body (200-400 words), and a single clear CTA. "
        "Do not include hashtags."
    ),
    "Twitter/X": (
        "FORMAT: STRICT HARD LIMIT of 280 characters for headline + body "
        "combined. Include 1-2 relevant hashtags. Every word must earn "
        "its place."
    ),
}

OUTPUT_FORMAT_INSTRUCTIONS = """\
Return your response as a single valid JSON object with EXACTLY these
fields and no others:
{
  "headline": "<a short hook / title / subject line>",
  "body": "<the main copy text>",
  "hashtags": ["<list of hashtag strings, each including the # symbol>"],
  "cta": "<the single call-to-action text>",
  "platform": "<the platform name exactly as given below>",
  "tone_used": "<the tone exactly as given below>",
  "character_count": <integer: total characters in headline + body combined>
}
Return ONLY the JSON object. No markdown formatting, no code fences,
no explanation before or after.\
"""


def build_master_template(product_name: str, description: str,
                           platform: str, tone: str) -> str:
    """Compile the final prompt sent to the model."""
    if platform not in VALID_PLATFORMS:
        raise ValueError(f"Unknown platform '{platform}'. Choose from {VALID_PLATFORMS}")
    if tone not in VALID_TONES:
        raise ValueError(f"Unknown tone '{tone}'. Choose from {VALID_TONES}")

    platform_rules = PLATFORM_INSTRUCTIONS[platform]

    return f"""You are a senior marketing copywriter.

{BRAND_RULES}

TASK
Product name: {product_name}
Product description (raw facts from the client): {description}
Target platform: {platform}
Required tone: {tone}

PLATFORM RULES
{platform_rules}

OUTPUT FORMAT
{OUTPUT_FORMAT_INSTRUCTIONS}
"""

Writing copywriter/templates.py


In [ ]:
%%writefile copywriter/params.py
"""
Inference parameter selection: Temperature, Top_P, and token limits.
"""

MODERN_MODELS = {"gpt-4o", "gpt-4o-mini", "o1", "o1-mini", "o3", "o3-mini"}


def get_temperature(platform: str, tone: str) -> float:
    """
    Low temperature (0.2) -> consistent, brand-safe, factual.
    High temperature (0.8) -> varied, surprising, engaging.
    """
    if tone == "Formal" or platform in ("Email", "LinkedIn"):
        return 0.2
    if tone == "Witty" or platform in ("Instagram", "Twitter/X"):
        return 0.8
    return 0.5


def get_max_tokens_kwarg(model: str, max_tokens: int = 400) -> dict:
    """
    Legacy models (gpt-4, gpt-3.5-turbo) use `max_tokens`.
    Modern reasoning models (gpt-4o, o1, ...) use
    `max_completion_tokens` — using the wrong one causes an API error.
    """
    if model in MODERN_MODELS:
        return {"max_completion_tokens": max(max_tokens, 256)}
    return {"max_tokens": max_tokens}

Writing copywriter/params.py


In [ ]:
from copywriter.params import get_temperature, get_max_tokens_kwarg

print(get_temperature("LinkedIn", "Professional"))   # 0.2 — brand-safe
print(get_temperature("Instagram", "Witty"))          # 0.8 — creative
print(get_max_tokens_kwarg("gpt-4"))                   # {'max_tokens': 400}
print(get_max_tokens_kwarg("gpt-4o"))                  # {'max_completion_tokens': 400}

0.2
0.8
{'max_tokens': 400}
{'max_completion_tokens': 400}


In [ ]:
%%writefile copywriter/csv_loader.py
"""CSV ingestion layer: read product variables from a spreadsheet."""
import csv
from pathlib import Path
from typing import Dict, List

REQUIRED_COLUMNS = {"product_name", "description", "platform", "tone"}


def load_products_from_csv(filepath: str) -> List[Dict]:
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"CSV file not found: {filepath}")

    products: List[Dict] = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"CSV is missing required columns: {missing}")

        for row in reader:
            products.append({
                "product_name": row["product_name"].strip(),
                "description": row["description"].strip(),
                "platform": row["platform"].strip(),
                "tone": row["tone"].strip(),
            })
    return products

Writing copywriter/csv_loader.py


In [ ]:
%%writefile data/sample_products.csv
product_name,description,platform,tone
NovaBrew Pro,AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycled materials.,LinkedIn,Professional
NovaBrew Pro,AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycled materials.,Instagram,Witty
ZenMat Flow,Smart yoga mat with posture-correction sensors and a companion app that tracks your practice over time.,Email,Formal
ZenMat Flow,Smart yoga mat with posture-correction sensors and a companion app that tracks your practice over time.,Twitter/X,Casual

Writing data/sample_products.csv


In [ ]:
from copywriter.csv_loader import load_products_from_csv

products = load_products_from_csv("data/sample_products.csv")
for p in products:
    print(p)

{'product_name': 'NovaBrew Pro', 'description': 'AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycled materials.', 'platform': 'LinkedIn', 'tone': 'Professional'}
{'product_name': 'NovaBrew Pro', 'description': 'AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycled materials.', 'platform': 'Instagram', 'tone': 'Witty'}
{'product_name': 'ZenMat Flow', 'description': 'Smart yoga mat with posture-correction sensors and a companion app that tracks your practice over time.', 'platform': 'Email', 'tone': 'Formal'}
{'product_name': 'ZenMat Flow', 'description': 'Smart yoga mat with posture-correction sensors and a companion app that tracks your practice over time.', 'platform': 'Twitter/X', 'tone': 'Casual'}


In [ ]:
%%writefile copywriter/realtime.py
"""
Real-time async pipeline.

Uses asyncio + httpx for concurrency, an asyncio.Semaphore to cap the
number of requests in flight at once, and Tenacity for automatic
retry with exponential backoff on transient failures.
"""
import asyncio
import json
import os
from typing import Dict, List, Union

import httpx
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential

from .params import get_max_tokens_kwarg, get_temperature
from .schemas import CopyOutput
from .templates import build_master_template

OPENAI_URL = "https://api.openai.com/v1/chat/completions"


def _build_payload(product: Dict, model: str) -> dict:
    prompt = build_master_template(
        product["product_name"], product["description"],
        product["platform"], product["tone"],
    )
    temperature = get_temperature(product["platform"], product["tone"])
    token_kwarg = get_max_tokens_kwarg(model)

    return {
        "model": model,
        "messages": [
            {"role": "system", "content": "You output only valid JSON."},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "response_format": {"type": "json_object"},
        **token_kwarg,
    }


@retry(
    retry=retry_if_exception_type((httpx.HTTPStatusError, httpx.TransportError)),
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=20),
)
async def _call_openai(client: httpx.AsyncClient, payload: dict, api_key: str) -> dict:
    response = await client.post(
        OPENAI_URL,
        headers={"Authorization": f"Bearer {api_key}"},
        json=payload,
        timeout=60.0,
    )
    response.raise_for_status()
    return response.json()


def _mock_response(product: Dict) -> dict:
    body = (
        f"{product['product_name']} brings a {product['tone'].lower()} "
        f"upgrade to your routine. {product['description'][:120]}"
    )
    hashtags = {
        "LinkedIn": ["#Innovation", "#Productivity"],
        "Instagram": ["#NewLaunch", "#MustHave", "#TechLife", "#Upgrade", "#DailyEssentials"],
        "Email": [],
        "Twitter/X": ["#NewLaunch"],
    }[product["platform"]]

    return {
        "headline": f"Meet {product['product_name']}",
        "body": body,
        "hashtags": hashtags,
        "cta": "Learn more today",
        "platform": product["platform"],
        "tone_used": product["tone"],
        "character_count": len(body) + len(f"Meet {product['product_name']}"),
    }


async def _generate_one(client, sem, product, model, api_key,
                         mock: bool = False) -> Union[CopyOutput, Exception]:
    try:
        if mock:
            await asyncio.sleep(0.3)
            data = _mock_response(product)
        else:
            payload = _build_payload(product, model)
            async with sem:
                raw = await _call_openai(client, payload, api_key)
            content = raw["choices"][0]["message"]["content"]
            data = json.loads(content)

        return CopyOutput(**data)
    except Exception as exc:
        return exc


async def realtime_pipeline(products: List[Dict], model: str = "gpt-4o",
                             max_concurrency: int = 5,
                             mock: bool = False) -> List[Union[CopyOutput, Exception]]:
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not mock and not api_key:
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Set it as an environment variable "
            "(or Colab secret), or pass mock=True to test without one."
        )

    sem = asyncio.Semaphore(max_concurrency)

    async with httpx.AsyncClient() as client:
        tasks = [
            _generate_one(client, sem, product, model, api_key, mock=mock)
            for product in products
        ]
        return await asyncio.gather(*tasks)

Writing copywriter/realtime.py


In [ ]:
from copywriter.realtime import realtime_pipeline

results = await realtime_pipeline(products, mock=True, max_concurrency=2)

for product, result in zip(products, results):
    print("=" * 50)
    if isinstance(result, Exception):
        print(f"{product['product_name']}: FAILED -> {result}")
        continue
    print(f"{product['product_name']} | {result.platform} | {result.tone_used}")
    print("Headline:", result.headline)
    print("Body:", result.body)
    print("Hashtags:", result.hashtags)
    print("CTA:", result.cta)

NovaBrew Pro | LinkedIn | Professional
Headline: Meet NovaBrew Pro
Body: NovaBrew Pro brings a professional upgrade to your routine. AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycle
Hashtags: ['#Innovation', '#Productivity']
CTA: Learn more today
NovaBrew Pro | Instagram | Witty
Headline: Meet NovaBrew Pro
Body: NovaBrew Pro brings a witty upgrade to your routine. AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycle
Hashtags: ['#NewLaunch', '#MustHave', '#TechLife', '#Upgrade', '#DailyEssentials']
CTA: Learn more today
ZenMat Flow | Email | Formal
Headline: Meet ZenMat Flow
Body: ZenMat Flow brings a formal upgrade to your routine. Smart yoga mat with posture-correction sensors and a companion app that tracks your practice over time.
Hashtags: []
CTA: Learn more today
ZenMat Flow | Twitter/X | Casual
Headline: Meet ZenMat Flow
Body: ZenMat Flow bri

In [ ]:
if "OPENAI_API_KEY" in os.environ:
    real_results = await realtime_pipeline([products[1]], model="gpt-4o", mock=False)
    r = real_results[0]
    if isinstance(r, Exception):
        print(f"API call failed: {r}")
    else:
        print(r.headline)
        print(r.body)
        print(r.hashtags)
else:
    print("Skipping — no API key set.")

API call failed: RetryError[<Future at 0x7d62d1e643b0 state=finished raised HTTPStatusError>]


In [ ]:
%%writefile copywriter/batch_pipeline.py
"""
Batch pipeline using the OpenAI Batch API.

Use this when results are NOT needed immediately. The Batch API
costs 50% less and runs on a separate, larger rate-limit pool, at
the cost of up to a 24-hour turnaround.
"""
import json
import os
import tempfile
import time
from pathlib import Path
from typing import Dict, List, Union

from .params import get_max_tokens_kwarg, get_temperature
from .realtime import _mock_response
from .schemas import CopyOutput
from .templates import build_master_template


def _build_batch_requests(products: List[Dict], model: str) -> List[dict]:
    requests = []
    for i, product in enumerate(products):
        prompt = build_master_template(
            product["product_name"], product["description"],
            product["platform"], product["tone"],
        )
        temperature = get_temperature(product["platform"], product["tone"])
        token_kwarg = get_max_tokens_kwarg(model)

        requests.append({
            "custom_id": f"product-{i}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model,
                "messages": [
                    {"role": "system", "content": "You output only valid JSON."},
                    {"role": "user", "content": prompt},
                ],
                "temperature": temperature,
                "response_format": {"type": "json_object"},
                **token_kwarg,
            },
        })
    return requests


def batch_pipeline(products: List[Dict], model: str = "gpt-4o",
                    poll_interval: int = 30,
                    mock: bool = False) -> List[Union[CopyOutput, Exception]]:
    if mock:
        results: List[Union[CopyOutput, Exception]] = []
        for product in products:
            try:
                results.append(CopyOutput(**_mock_response(product)))
            except Exception as exc:
                results.append(exc)
        return results

    from openai import OpenAI

    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set.")

    client = OpenAI(api_key=api_key)
    requests = _build_batch_requests(products, model)

    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".jsonl", delete=False, encoding="utf-8"
    ) as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
        jsonl_path = f.name

    with open(jsonl_path, "rb") as f:
        batch_file = client.files.create(file=f, purpose="batch")
    Path(jsonl_path).unlink(missing_ok=True)

    batch = client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    while batch.status not in ("completed", "failed", "cancelled", "expired"):
        time.sleep(poll_interval)
        batch = client.batches.retrieve(batch.id)
        print(f"Batch status: {batch.status}")

    if batch.status != "completed":
        raise RuntimeError(f"Batch ended with status: {batch.status}")

    output_file = client.files.content(batch.output_file_id)
    raw_by_id = {
        json.loads(line)["custom_id"]: json.loads(line)
        for line in output_file.text.splitlines()
    }

    results = []
    for i in range(len(products)):
        record = raw_by_id.get(f"product-{i}")
        try:
            content = record["response"]["body"]["choices"][0]["message"]["content"]
            data = json.loads(content)
            results.append(CopyOutput(**data))
        except Exception as exc:
            results.append(exc)

    return results

Writing copywriter/batch_pipeline.py


In [ ]:
from copywriter.batch_pipeline import batch_pipeline

batch_results = batch_pipeline(products, mock=True)
for product, result in zip(products, batch_results):
    status = "OK" if not isinstance(result, Exception) else f"FAILED: {result}"
    print(f"{product['product_name']} ({product['platform']}): {status}")

NovaBrew Pro (LinkedIn): OK
NovaBrew Pro (Instagram): OK
ZenMat Flow (Email): OK
ZenMat Flow (Twitter/X): OK


In [ ]:
%%writefile run.py
#!/usr/bin/env python3
"""
Project 2: Automated Copywriting & Tone Transformer
Command-line entry point.
"""
import argparse
import asyncio
import sys

from copywriter.batch_pipeline import batch_pipeline
from copywriter.csv_loader import load_products_from_csv
from copywriter.realtime import realtime_pipeline
from copywriter.templates import VALID_PLATFORMS, VALID_TONES


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Automated Copywriting & Tone Transformer",
        prefix_chars="-+",
    )
    parser.add_argument("--product", help="Product name")
    parser.add_argument("--description", default="", help="Raw product description")
    parser.add_argument("--platform", choices=VALID_PLATFORMS, help="Target platform")
    parser.add_argument("--tone", choices=VALID_TONES, default="Professional")
    parser.add_argument("--model", default="gpt-4o", help="Model name (default: gpt-4o)")
    parser.add_argument("--concurrency", type=int, default=5,
                         help="Max concurrent requests in real-time mode")
    parser.add_argument("--csv", default="", help="Path to a CSV file of products")
    parser.add_argument("+b", "--batch", action="store_true",
                         help="Use the OpenAI Batch API (cheaper, up to 24h)")
    parser.add_argument("--mock", action="store_true",
                         help="Run the full pipeline without calling the real API")
    return parser


def main() -> None:
    args = build_parser().parse_args()

    if args.csv:
        products = load_products_from_csv(args.csv)
    elif args.product and args.platform:
        products = [{
            "product_name": args.product,
            "description": args.description,
            "platform": args.platform,
            "tone": args.tone,
        }]
    else:
        print("Provide either --csv <file>  OR  both --product and --platform.")
        sys.exit(1)

    if args.batch:
        print(f"Running BATCH pipeline for {len(products)} product(s)...")
        results = batch_pipeline(products, model=args.model, mock=args.mock)
    else:
        print(f"Running REAL-TIME pipeline for {len(products)} product(s)...")
        results = asyncio.run(
            realtime_pipeline(products, model=args.model,
                               max_concurrency=args.concurrency, mock=args.mock)
        )

    for i, result in enumerate(results):
        print("\n" + "=" * 60)
        if isinstance(result, Exception):
            print(f"Product {i + 1} ({products[i]['product_name']}): "
                  f"FAILED -> {result}")
            continue

        print(f"Product {i + 1}: {products[i]['product_name']} | "
              f"{result.platform} | {result.tone_used}")
        print("-" * 60)
        print(f"Headline: {result.headline}")
        print(f"Body:\n{result.body}")
        if result.hashtags:
            print(f"Hashtags: {' '.join(result.hashtags)}")
        print(f"CTA: {result.cta}")
        print(f"Character count: {result.character_count}")


if __name__ == "__main__":
    main()

Writing run.py


In [ ]:
!python run.py --csv data/sample_products.csv --mock

Running REAL-TIME pipeline for 4 product(s)...

Product 1: NovaBrew Pro | LinkedIn | Professional
------------------------------------------------------------
Headline: Meet NovaBrew Pro
Body:
NovaBrew Pro brings a professional upgrade to your routine. AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycle
Hashtags: #Innovation #Productivity
CTA: Learn more today
Character count: 197

Product 2: NovaBrew Pro | Instagram | Witty
------------------------------------------------------------
Headline: Meet NovaBrew Pro
Body:
NovaBrew Pro brings a witty upgrade to your routine. AI-powered coffee maker that learns your taste preferences and brews in 30 seconds. App-controlled and made from recycle
Hashtags: #NewLaunch #MustHave #TechLife #Upgrade #DailyEssentials
CTA: Learn more today
Character count: 190

Product 3: ZenMat Flow | Email | Formal
------------------------------------------------------------
Headline: Meet ZenMat

In [ ]:
!python run.py --product "ZenMat Flow" --platform Email --tone Formal \
  --description "Smart yoga mat with posture-correction sensors" --mock

Running REAL-TIME pipeline for 1 product(s)...

Product 1: ZenMat Flow | Email | Formal
------------------------------------------------------------
Headline: Meet ZenMat Flow
Body:
ZenMat Flow brings a formal upgrade to your routine. Smart yoga mat with posture-correction sensors
CTA: Learn more today
Character count: 115


In [ ]:
%%writefile requirements.txt
openai>=1.40.0
httpx>=0.27.0
tenacity>=8.2.0
pydantic>=2.0.0
python-dotenv>=1.0.0

Writing requirements.txt


In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
*.egg-info/
.venv/
venv/
.env
.ipynb_checkpoints/
.DS_Store

Writing .gitignore


In [ ]:
%%writefile .env.example
# Copy this file to .env and fill in your real key.
# NEVER commit your real .env file to GitHub.
OPENAI_API_KEY=your_openai_api_key_here

Writing .env.example


In [ ]:
%%writefile README.md
# Automated Copywriting & Tone Transformer

A Python pipeline that generates platform-specific, tone-matched marketing
copy from raw product facts, using dynamic prompt templates, tuned
inference parameters, and Pydantic-validated output.

## Setup
```bash
pip install -r requirements.txt
export OPENAI_API_KEY=sk-...
```

## Usage
```bash
# Mock mode (no API key needed)
python run.py --csv data/sample_products.csv --mock

# Real-time pipeline
python run.py --product "NovaBrew Pro" --platform Instagram --tone Witty \
  --description "AI coffee maker, 30-second brew"

# Batch API (50% cheaper, up to 24h)
python run.py --csv data/sample_products.csv +b
```

Writing README.md


In [ ]:
!find . -path ./sample_data -prune -o -type f -print | grep -v __pycache__

./.config/.last_update_check.json
./.config/configurations/config_default
./.config/gce
./.config/.last_survey_prompt.yaml
./.config/logs/2026.06.04/13.32.21.210570.log
./.config/logs/2026.06.04/13.32.38.346437.log
./.config/logs/2026.06.04/13.31.42.499627.log
./.config/logs/2026.06.04/13.32.02.654775.log
./.config/logs/2026.06.04/13.32.18.735754.log
./.config/logs/2026.06.04/13.32.39.344962.log
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/config_sentinel
./.config/active_config
./.config/.last_opt_in_prompt.yaml
./.config/default_configs.db
./.gitignore
./requirements.txt
./.env.example
./copywriter/realtime.py
./copywriter/batch_pipeline.py
./copywriter/csv_loader.py
./copywriter/__init__.py
./copywriter/templates.py
./copywriter/params.py
./copywriter/schemas.py
./README.md
./run.py
./data/sample_products.csv


In [ ]:
!find . -path ./sample_data -prune -o -type f -print | grep -v __pycache__

./.config/.last_update_check.json
./.config/configurations/config_default
./.config/gce
./.config/.last_survey_prompt.yaml
./.config/logs/2026.06.04/13.32.21.210570.log
./.config/logs/2026.06.04/13.32.38.346437.log
./.config/logs/2026.06.04/13.31.42.499627.log
./.config/logs/2026.06.04/13.32.02.654775.log
./.config/logs/2026.06.04/13.32.18.735754.log
./.config/logs/2026.06.04/13.32.39.344962.log
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/config_sentinel
./.config/active_config
./.config/.last_opt_in_prompt.yaml
./.config/default_configs.db
./.gitignore
./requirements.txt
./.env.example
./copywriter/realtime.py
./copywriter/batch_pipeline.py
./copywriter/csv_loader.py
./copywriter/__init__.py
./copywriter/templates.py
./copywriter/params.py
./copywriter/schemas.py
./README.md
./run.py
./data/sample_products.csv


In [ ]:
!ls -R copywriter data
!ls *.txt *.md .gitignore *.py 2>/dev/null

copywriter:
batch_pipeline.py  __init__.py	__pycache__  schemas.py
csv_loader.py	   params.py	realtime.py  templates.py

copywriter/__pycache__:
batch_pipeline.cpython-312.pyc	realtime.cpython-312.pyc
csv_loader.cpython-312.pyc	schemas.cpython-312.pyc
__init__.cpython-312.pyc	templates.cpython-312.pyc
params.cpython-312.pyc

data:
sample_products.csv
.gitignore  README.md  requirements.txt  run.py


In [57]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") # Changed to use a more descriptive secret name
GITHUB_USERNAME = "akashsb2005"     # <- replace with YOUR GitHub username
REPO_NAME = "copywriting-tone-transformer"   # <- must match the repo name you created

remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print("Remote configured.")

Remote configured.


In [58]:
!git init -q
!git config user.email "bagojiakash75@gmail.com"
!git config user.name "akashsb2005"
!git add .
!git commit -q -m "akashsb2005"
!git branch -M main

On branch main
nothing to commit, working tree clean


In [60]:
import subprocess
subprocess.run(["git", "remote", "remove", "origin"], capture_output=True)
subprocess.run(["git", "remote", "add", "origin", remote_url], capture_output=True)

!git push -u "{remote_url}" main

To https://github.com/akashsb2005/copywriting-tone-transformer.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/akashsb2005/copywriting-tone-transformer.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [65]:
import subprocess

# Remove any existing remote to ensure a clean setup for force push
subprocess.run(["git", "remote", "remove", "origin"], capture_output=True)
# Add the remote with the correct URL, which includes the GITHUB_TOKEN
subprocess.run(["git", "remote", "add", "origin", remote_url], capture_output=True)

# Force push the local main branch to the remote main branch.
# This overwrites the remote history with the local history, which is common
# when initializing a GitHub repo from an existing local project.
print("Force pushing local changes to remote...")
!git push -u origin main --force

Force pushing local changes to remote...
Enumerating objects: 43, done.
Counting objects: 100% (43/43), done.
Delta compression using up to 2 threads
Compressing objects: 100% (35/35), done.
Writing objects: 100% (43/43), 8.43 MiB | 1.54 MiB/s, done.
Total 43 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), done.
To https://github.com/akashsb2005/copywriting-tone-transformer.git
 + de16b03...fe2a511 main -> main (forced update)
Branch 'main' set up to track remote branch 'main' from 'origin'.


In [ ]:
!mkdir -p notebooks
from google.colab import files
uploaded = files.upload()